# FINAL REPORT (Phần bổ sung): Hyperparameter Tuning
## Credit Scoring — Give Me Some Credit

**Sinh viên**: QuangMinh  
**Khóa**: MSE35HN  
**Ngày**: 2026-03-13

---

## Mục đích

Notebook này **bổ sung** cho [05_Final_Report.ipynb](05_Final_Report.ipynb) với kết quả **Hyperparameter Tuning** từ Phase 4.2.

### Nội dung:
1. Tóm tắt kết quả Phase 3 & 4 (5 models default)
2. Kết quả Phase 4.2 (LightGBM & XGBoost tuned)
3. **So sánh tổng hợp**: 5 default + 2 tuned = 7 entries
4. Phân tích: Tuning thay đổi gì?
5. Xếp hạng cuối cùng (cập nhật)
6. Kết luận bổ sung

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import (
    roc_auc_score, recall_score, f1_score, precision_score,
    classification_report, confusion_matrix, roc_curve
)

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

from datetime import datetime
print(f"Final Report (Tuning) generated at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

report_dir = '../reports/[5.2] Final Report Tuning/'
os.makedirs(report_dir, exist_ok=True)

Final Report (Tuning) generated at: 2026-03-13 20:33:25


## 1. Tóm tắt kết quả trước Tuning (Phase 3 & 4)

In [2]:
print("="*80)
print("KẾT QUẢ PHASE 3 — 5 Models với Default Parameters")
print("="*80)

phase3 = pd.read_csv('../reports/[3] Ket qua Model Building/model_comparison.csv')
print()
display(phase3.style.format({
    'AUC-ROC': '{:.4f}', 'Recall': '{:.4f}',
    'F1-Score': '{:.4f}', 'Precision': '{:.4f}',
    'Train Time (s)': '{:.2f}'
}).highlight_max(subset=['AUC-ROC', 'Recall'], color='lightgreen'))

print("\nNhận xét:")
print("• LightGBM dẫn đầu AUC (0.8672) và Recall (0.7711)")
print("• XGBoost xếp #4 về AUC (0.8506) — thấp hơn cả LR (0.8610)")
print("• Tất cả dùng default parameters → tiềm năng cải thiện qua tuning")

KẾT QUẢ PHASE 3 — 5 Models với Default Parameters



,Model,AUC-ROC,Recall,F1-Score,Precision,Train Time (s)
0,LightGBM,0.8672,0.7711,0.3428,0.2204,0.24
1,Logistic Regression,0.8610,0.7616,0.3356,0.2152,1.47
2,LinearSVC,0.8574,0.0658,0.1182,0.5789,0.36
3,XGBoost,0.8506,0.7042,0.3534,0.2359,0.22
4,Random Forest,0.8335,0.1671,0.2566,0.5528,1.54



Nhận xét:
• LightGBM dẫn đầu AUC (0.8672) và Recall (0.7711)
• XGBoost xếp #4 về AUC (0.8506) — thấp hơn cả LR (0.8610)
• Tất cả dùng default parameters → tiềm năng cải thiện qua tuning


## 2. Kết quả Phase 4.2 — Hyperparameter Tuning

### 2.1 Cấu hình Tuning

| Thông số | Giá trị |
|----------|--------|
| Method | RandomizedSearchCV |
| n_iter | 20 (thử 20 random combinations) |
| cv | 3-fold StratifiedKFold |
| scoring | roc_auc |
| n_jobs | 2 (tránh đơ máy) |
| Models tuned | LightGBM, XGBoost |

In [3]:
print("="*80)
print("BEST HYPERPARAMETERS TÌM ĐƯỢC")
print("="*80)

best_params = pd.read_csv('../reports/[4.2] Ket qua Tuning/best_params.csv')
print()

# LightGBM params
lgbm_params = best_params[best_params['Model'] == 'LightGBM'].iloc[0]
xgb_params = best_params[best_params['Model'] == 'XGBoost'].iloc[0]

params_compare = pd.DataFrame([
    {'Parameter': 'n_estimators', 'LightGBM Default': 100, 'LightGBM Tuned': int(lgbm_params['n_estimators']), 'XGBoost Default': 100, 'XGBoost Tuned': int(xgb_params['n_estimators'])},
    {'Parameter': 'learning_rate', 'LightGBM Default': 0.1, 'LightGBM Tuned': lgbm_params['learning_rate'], 'XGBoost Default': 0.3, 'XGBoost Tuned': xgb_params['learning_rate']},
    {'Parameter': 'max_depth', 'LightGBM Default': -1, 'LightGBM Tuned': int(lgbm_params['max_depth']), 'XGBoost Default': 6, 'XGBoost Tuned': int(xgb_params['max_depth'])},
    {'Parameter': 'subsample', 'LightGBM Default': 1.0, 'LightGBM Tuned': lgbm_params['subsample'], 'XGBoost Default': 1.0, 'XGBoost Tuned': xgb_params['subsample']},
    {'Parameter': 'Best CV AUC', 'LightGBM Default': '—', 'LightGBM Tuned': round(lgbm_params['Best_CV_AUC'], 4), 'XGBoost Default': '—', 'XGBoost Tuned': round(xgb_params['Best_CV_AUC'], 4)},
])

display(params_compare)

print("\nPattern chung: learning_rate ↓ + n_estimators ↑ + max_depth = 3")
print("→ Học chậm hơn + nhiều trees hơn + shallow trees = tốt hơn")

BEST HYPERPARAMETERS TÌM ĐƯỢC



,Parameter,LightGBM Default,LightGBM Tuned,XGBoost Default,XGBoost Tuned
0,n_estimators,100,300.0000,100,500.0000
1,learning_rate,0.1,0.0500,0.3,0.0500
2,max_depth,-1,3.0000,6,3.0000
3,subsample,1.0,0.8000,1.0,0.8000
4,Best CV AUC,—,0.8647,—,0.8641



Pattern chung: learning_rate ↓ + n_estimators ↑ + max_depth = 3
→ Học chậm hơn + nhiều trees hơn + shallow trees = tốt hơn


In [4]:
print("="*80)
print("SO SÁNH TRỰC TIẾP: Default vs Tuned")
print("="*80)

dvt = pd.read_csv('../reports/[4.2] Ket qua Tuning/default_vs_tuned.csv')
print()
display(dvt[['Model', 'AUC_Default', 'AUC_Tuned', 'AUC_Change',
             'Recall_Default', 'Recall_Tuned', 'Recall_Change',
             'F1_Default', 'F1_Tuned', 'F1_Change']].style.format({
    'AUC_Default': '{:.4f}', 'AUC_Tuned': '{:.4f}', 'AUC_Change': '{:+.4f}',
    'Recall_Default': '{:.4f}', 'Recall_Tuned': '{:.4f}', 'Recall_Change': '{:+.4f}',
    'F1_Default': '{:.4f}', 'F1_Tuned': '{:.4f}', 'F1_Change': '{:+.4f}'
}))

print("\nNhận xét:")
print(f"• LightGBM: AUC +{dvt.iloc[0]['AUC_Change']:.4f}, Recall +{dvt.iloc[0]['Recall_Change']:.4f} — cải thiện nhẹ (đã gần ceiling)")
print(f"• XGBoost:  AUC +{dvt.iloc[1]['AUC_Change']:.4f}, Recall +{dvt.iloc[1]['Recall_Change']:.4f} — cải thiện đáng kể!")
print(f"• XGBoost nhảy từ AUC #4 (0.8506) lên gần bằng LightGBM (0.8689 vs 0.8695)")

SO SÁNH TRỰC TIẾP: Default vs Tuned



,Model,AUC_Default,AUC_Tuned,AUC_Change,Recall_Default,Recall_Tuned,Recall_Change,F1_Default,F1_Tuned,F1_Change
0,LightGBM,0.8672,0.8695,+0.0023,0.7711,0.7900,+0.0190,0.3428,0.3360,-0.0068
1,XGBoost,0.8506,0.8689,+0.0182,0.7042,0.7796,+0.0753,0.3534,0.3402,-0.0133



Nhận xét:
• LightGBM: AUC +0.0023, Recall +0.0190 — cải thiện nhẹ (đã gần ceiling)
• XGBoost:  AUC +0.0182, Recall +0.0753 — cải thiện đáng kể!
• XGBoost nhảy từ AUC #4 (0.8506) lên gần bằng LightGBM (0.8689 vs 0.8695)


## 3. So sánh tổng hợp: 5 Default + 2 Tuned

In [5]:
print("="*80)
print("BẢNG XẾP HẠNG TỔNG HỢP — 7 Entries")
print("="*80)

tuning_comp = pd.read_csv('../reports/[4.2] Ket qua Tuning/tuning_comparison.csv')
tuning_comp = tuning_comp.sort_values('AUC-ROC', ascending=False).reset_index(drop=True)
tuning_comp.index += 1
tuning_comp.index.name = 'Rank'

print()
display(tuning_comp.style.format({
    'AUC-ROC': '{:.4f}', 'Recall': '{:.4f}',
    'F1-Score': '{:.4f}', 'Precision': '{:.4f}'
}).apply(lambda x: ['background-color: #d4edda' if v == 'Tuned' else '' for v in x], subset=['Version'])
 .apply(lambda x: ['font-weight: bold' if v == 'Tuned' else '' for v in x], subset=['Version']))

BẢNG XẾP HẠNG TỔNG HỢP — 7 Entries



,Model,Version,AUC-ROC,Recall,F1-Score,Precision
Rank,,,,,,
1,LightGBM,Tuned,0.8695,0.7900,0.3360,0.2134
2,XGBoost,Tuned,0.8689,0.7796,0.3402,0.2175
3,LightGBM,Default,0.8672,0.7711,0.3428,0.2204
4,Logistic Regression,Default,0.8610,0.7616,0.3356,0.2152
5,LinearSVC,Default,0.8574,0.0658,0.1182,0.5789
6,XGBoost,Default,0.8506,0.7042,0.3534,0.2359
7,Random Forest,Default,0.8335,0.1671,0.2566,0.5528


In [ ]:
# 3.0 So sánh tổng hợp — Biểu đồ tách riêng
tuning_images = [
    ('../reports/[4.2] Ket qua Tuning/roc_default_vs_tuned.png', '3.0 Tuning — ROC Curves: Default vs Tuned'),
    ('../reports/[4.2] Ket qua Tuning/metrics_default_vs_tuned.png', '3.0 Tuning — Metrics: Default vs Tuned')
]

for path, title in tuning_images:
    fig, ax = plt.subplots(figsize=(16, 9))
    img = mpimg.imread(path)
    ax.imshow(img)
    ax.set_title(title, fontsize=16, fontweight='bold', pad=15)
    ax.axis('off')
    plt.tight_layout()
    
    save_name = f"tuning_{os.path.basename(path)}"
    plt.savefig(report_dir + save_name, dpi=200, bbox_inches='tight')
    plt.show()
    print(f"  → Saved: {save_name}\n")

## 4. Phân tích: Tuning thay đổi gì?

In [ ]:
# 4.0 Phân tích Tuning — Confusion Matrices tách riêng
data_dir = '../data/processed/'
X_val = pd.read_csv(data_dir + 'X_val.csv')
y_val = pd.read_csv(data_dir + 'y_val.csv').squeeze()

lgbm_default = joblib.load('../models/lightgbm.pkl')
lgbm_tuned = joblib.load('../models/lightgbm_tuned.pkl')
xgb_default = joblib.load('../models/xgboost.pkl')
xgb_tuned = joblib.load('../models/xgboost_tuned.pkl')

models_to_compare = [
    (lgbm_default, 'LightGBM Default', 'Blues', '4.0 Tuning — Confusion Matrix: LightGBM Default'),
    (lgbm_tuned, 'LightGBM Tuned', 'Greens', '4.0 Tuning — Confusion Matrix: LightGBM Tuned'),
    (xgb_default, 'XGBoost Default', 'Oranges', '4.0 Tuning — Confusion Matrix: XGBoost Default'),
    (xgb_tuned, 'XGBoost Tuned', 'RdPu', '4.0 Tuning — Confusion Matrix: XGBoost Tuned'),
]

for model, name, cmap, title in models_to_compare:
    fig, ax = plt.subplots(figsize=(8, 7))
    
    y_pred = model.predict(X_val)
    y_prob = model.predict_proba(X_val)[:, 1]
    cm = confusion_matrix(y_val, y_pred)
    auc = roc_auc_score(y_val, y_prob)
    recall = recall_score(y_val, y_pred)
    
    cm_pct = cm / cm.sum() * 100
    annot = [[f"{cm[r][c]:,}\n({cm_pct[r][c]:.1f}%)" for c in range(2)] for r in range(2)]
    
    sns.heatmap(cm, annot=annot, fmt='', cmap=cmap, ax=ax,
                xticklabels=['Good', 'Bad'], yticklabels=['Good', 'Bad'],
                cbar=False, linewidths=1, linecolor='white',
                annot_kws={'fontsize': 14})
    ax.set_title(f"{title}\nAUC={auc:.4f} | Recall={recall:.4f}", fontsize=14, fontweight='bold', pad=15)
    ax.set_ylabel('Thực tế', fontsize=12)
    ax.set_xlabel('Dự đoán', fontsize=12)
    
    plt.tight_layout()
    save_name = f"cm_{name.lower().replace(' ', '_')}.png"
    plt.savefig(report_dir + save_name, dpi=200, bbox_inches='tight')
    plt.show()
    print(f"  → Saved: {save_name}\n")

In [8]:
# Phân tích chi tiết: bao nhiêu khách xấu được phát hiện thêm?
print("="*80)
print("PHÂN TÍCH CHI TIẾT: TUNING CẢI THIỆN BAO NHIÊU?")
print("="*80)

total_bad = (y_val == 1).sum()

for name, model_d, model_t in [
    ('LightGBM', lgbm_default, lgbm_tuned),
    ('XGBoost', xgb_default, xgb_tuned)
]:
    pred_d = model_d.predict(X_val)
    pred_t = model_t.predict(X_val)
    prob_d = model_d.predict_proba(X_val)[:, 1]
    prob_t = model_t.predict_proba(X_val)[:, 1]
    
    tp_d = ((y_val == 1) & (pred_d == 1)).sum()
    tp_t = ((y_val == 1) & (pred_t == 1)).sum()
    fn_d = ((y_val == 1) & (pred_d == 0)).sum()
    fn_t = ((y_val == 1) & (pred_t == 0)).sum()
    fp_d = ((y_val == 0) & (pred_d == 1)).sum()
    fp_t = ((y_val == 0) & (pred_t == 1)).sum()
    
    print(f"\n{'─'*60}")
    print(f"  {name}")
    print(f"{'─'*60}")
    print(f"  Tổng khách xấu (Bad) trong validation: {total_bad:,}")
    print(f"")
    print(f"  {'Metric':<30} {'Default':>10} {'Tuned':>10} {'Change':>10}")
    print(f"  {'─'*60}")
    print(f"  {'AUC-ROC':<30} {roc_auc_score(y_val, prob_d):>10.4f} {roc_auc_score(y_val, prob_t):>10.4f} {roc_auc_score(y_val, prob_t) - roc_auc_score(y_val, prob_d):>+10.4f}")
    print(f"  {'Bad phát hiện đúng (TP)':<30} {tp_d:>10,} {tp_t:>10,} {tp_t - tp_d:>+10,}")
    print(f"  {'Bad bỏ sót (FN)':<30} {fn_d:>10,} {fn_t:>10,} {fn_t - fn_d:>+10,}")
    print(f"  {'Good báo sai (FP)':<30} {fp_d:>10,} {fp_t:>10,} {fp_t - fp_d:>+10,}")
    print(f"  {'Recall':<30} {tp_d/total_bad:>10.4f} {tp_t/total_bad:>10.4f} {(tp_t-tp_d)/total_bad:>+10.4f}")

PHÂN TÍCH CHI TIẾT: TUNING CẢI THIỆN BAO NHIÊU?

────────────────────────────────────────────────────────────
  LightGBM
────────────────────────────────────────────────────────────
  Tổng khách xấu (Bad) trong validation: 2,005

  Metric                            Default      Tuned     Change
  ────────────────────────────────────────────────────────────
  AUC-ROC                            0.8672     0.8695    +0.0023
  Bad phát hiện đúng (TP)             1,546      1,584        +38
  Bad bỏ sót (FN)                       459        421        -38
  Good báo sai (FP)                   5,469      5,840       +371
  Recall                             0.7711     0.7900    +0.0190

────────────────────────────────────────────────────────────
  XGBoost
────────────────────────────────────────────────────────────
  Tổng khách xấu (Bad) trong validation: 2,005

  Metric                            Default      Tuned     Change
  ────────────────────────────────────────────────────────────
 

## 5. Xếp hạng cuối cùng (Cập nhật sau Tuning)

In [9]:
print("="*80)
print("XẾP HẠNG CUỐI CÙNG — SAU HYPERPARAMETER TUNING")
print("="*80)

# Tính metrics cho tất cả models
X_val_scaled = pd.read_csv(data_dir + 'X_val_scaled.csv')

all_models = {
    'LightGBM (Tuned)': (lgbm_tuned, X_val),
    'XGBoost (Tuned)': (xgb_tuned, X_val),
    'LightGBM (Default)': (lgbm_default, X_val),
    'Logistic Regression': (joblib.load('../models/logistic_regression.pkl'), X_val_scaled),
    'LinearSVC': (joblib.load('../models/linear_svc.pkl'), X_val_scaled),
    'XGBoost (Default)': (xgb_default, X_val),
    'Random Forest': (joblib.load('../models/random_forest.pkl'), X_val),
}

final_ranking = []
for name, (model, X) in all_models.items():
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1]
    final_ranking.append({
        'Model': name,
        'AUC-ROC': roc_auc_score(y_val, y_prob),
        'Recall': recall_score(y_val, y_pred),
        'F1-Score': f1_score(y_val, y_pred),
        'Precision': precision_score(y_val, y_pred),
    })

final_df = pd.DataFrame(final_ranking).sort_values('AUC-ROC', ascending=False).reset_index(drop=True)
final_df.index += 1
final_df.index.name = 'Rank'

print()
display(final_df.style.format({
    'AUC-ROC': '{:.4f}', 'Recall': '{:.4f}',
    'F1-Score': '{:.4f}', 'Precision': '{:.4f}'
}).apply(lambda x: ['background-color: #d4edda' if '(Tuned)' in str(v) else '' for v in x], subset=['Model']))

# Save
final_df.to_csv(report_dir + 'final_ranking_with_tuning.csv')
print("\nSaved: final_ranking_with_tuning.csv")

XẾP HẠNG CUỐI CÙNG — SAU HYPERPARAMETER TUNING



,Model,AUC-ROC,Recall,F1-Score,Precision
Rank,,,,,
1,LightGBM (Tuned),0.8695,0.7900,0.3360,0.2134
2,XGBoost (Tuned),0.8689,0.7796,0.3402,0.2175
3,LightGBM (Default),0.8672,0.7711,0.3428,0.2204
4,Logistic Regression,0.8610,0.7616,0.3356,0.2152
5,LinearSVC,0.8574,0.0658,0.1182,0.5789
6,XGBoost (Default),0.8506,0.7042,0.3534,0.2359
7,Random Forest,0.8335,0.1671,0.2566,0.5528



Saved: final_ranking_with_tuning.csv


In [ ]:
# 5.0 Xếp hạng cuối cùng — ROC Curves tất cả 7 models
fig, ax = plt.subplots(1, 1, figsize=(14, 10))

colors = {
    'LightGBM (Tuned)': '#e74c3c',
    'XGBoost (Tuned)': '#f39c12',
    'LightGBM (Default)': '#e74c3c',
    'XGBoost (Default)': '#f39c12',
    'Logistic Regression': '#3498db',
    'LinearSVC': '#9b59b6',
    'Random Forest': '#2ecc71',
}
linestyles = {
    'LightGBM (Tuned)': '-',
    'XGBoost (Tuned)': '-',
    'LightGBM (Default)': '--',
    'XGBoost (Default)': '--',
    'Logistic Regression': ':',
    'LinearSVC': ':',
    'Random Forest': ':',
}

for name, (model, X) in all_models.items():
    y_prob = model.predict_proba(X)[:, 1]
    fpr, tpr, _ = roc_curve(y_val, y_prob)
    auc = roc_auc_score(y_val, y_prob)
    lw = 2.5 if '(Tuned)' in name else 1.5
    ax.plot(fpr, tpr, color=colors[name], linestyle=linestyles[name], lw=lw,
            label=f"{name} (AUC={auc:.4f})")

ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.3)
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate (Recall)', fontsize=12)
ax.set_title('5.0 Xếp hạng — ROC Curves: All Models (Default + Tuned)', fontsize=16, fontweight='bold', pad=15)
ax.legend(loc='lower right', fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(report_dir + 'roc_all_models.png', dpi=200, bbox_inches='tight')
plt.show()
print("  → Saved: roc_all_models.png")

## 6. Kết luận bổ sung

In [11]:
best = final_df.iloc[0]
xgb_t = final_df[final_df['Model'] == 'XGBoost (Tuned)'].iloc[0]

print("="*80)
print("KẾT LUẬN BỔ SUNG SAU HYPERPARAMETER TUNING")
print("="*80)

print(f"""
╔══════════════════════════════════════════════════════════════════════╗
║              XẾP HẠNG CUỐI CÙNG (SAU TUNING)                      ║
╠══════╦══════════════════════════╦════════════╦═════════════════════╣
║ Hạng ║ Model                    ║ AUC-ROC    ║ Recall              ║
╠══════╬══════════════════════════╬════════════╬═════════════════════╣
║  1   ║ LightGBM (Tuned)        ║ {best['AUC-ROC']:.4f}     ║ {best['Recall']:.4f}              ║
║  2   ║ XGBoost (Tuned)         ║ {xgb_t['AUC-ROC']:.4f}     ║ {xgb_t['Recall']:.4f}              ║
╚══════╩══════════════════════════╩════════════╩═════════════════════╝

1. BEST MODEL VẪN LÀ LightGBM (Tuned):
   • AUC-ROC = {best['AUC-ROC']:.4f} (từ 0.8672, +0.0023)
   • Recall = {best['Recall']:.4f} (phát hiện ~79% khách hàng xấu)

2. XGBoost CẢI THIỆN ĐÁNG KỂ SAU TUNING:
   • AUC: 0.8506 → {xgb_t['AUC-ROC']:.4f} (+0.0183)
   • Từ #4 (thấp hơn LR) → #2 (gần bằng LightGBM)
   • Nguyên nhân: default learning_rate=0.3 quá cao

3. PHÁT HIỆN QUAN TRỌNG:
   • Default params có thể khiến model bị đánh giá SAI năng lực
   • XGBoost default AUC=0.8506 < LR AUC=0.8610
   • XGBoost tuned AUC={xgb_t['AUC-ROC']:.4f} > LR AUC=0.8610
   → Hyperparameter Tuning THỰC SỰ QUAN TRỌNG

4. PATTERN TỐI ƯU (cả 2 models):
   • learning_rate ↓ (0.1-0.3 → 0.05)
   • n_estimators ↑ (100 → 300-500)
   • max_depth = 3 (shallow trees, tránh overfit)
   • subsample = 0.8 (regularization)

5. KHUYẾN NGHỊ CẬP NHẬT:
   • Dùng LightGBM (Tuned) làm model chính
   • Dùng XGBoost (Tuned) làm model backup
   • Logistic Regression vẫn phù hợp cho explainability/audit
""")

KẾT LUẬN BỔ SUNG SAU HYPERPARAMETER TUNING

╔══════════════════════════════════════════════════════════════════════╗
║              XẾP HẠNG CUỐI CÙNG (SAU TUNING)                      ║
╠══════╦══════════════════════════╦════════════╦═════════════════════╣
║ Hạng ║ Model                    ║ AUC-ROC    ║ Recall              ║
╠══════╬══════════════════════════╬════════════╬═════════════════════╣
║  1   ║ LightGBM (Tuned)        ║ 0.8695     ║ 0.7900              ║
║  2   ║ XGBoost (Tuned)         ║ 0.8689     ║ 0.7796              ║
╚══════╩══════════════════════════╩════════════╩═════════════════════╝

1. BEST MODEL VẪN LÀ LightGBM (Tuned):
   • AUC-ROC = 0.8695 (từ 0.8672, +0.0023)
   • Recall = 0.7900 (phát hiện ~79% khách hàng xấu)

2. XGBoost CẢI THIỆN ĐÁNG KỂ SAU TUNING:
   • AUC: 0.8506 → 0.8689 (+0.0183)
   • Từ #4 (thấp hơn LR) → #2 (gần bằng LightGBM)
   • Nguyên nhân: default learning_rate=0.3 quá cao

3. PHÁT HIỆN QUAN TRỌNG:
   • Default params có thể khiến model bị đánh 

In [12]:
print("\n" + "═" * 80)
print("   DỰ ÁN HOÀN THÀNH — Give Me Some Credit — Credit Scoring")
print("═" * 80)
print(f"\n  Thời gian: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\n  Phases đã thực hiện:")
print("    ✅ Phase 1: EDA")
print("    ✅ Phase 2: Preprocessing & Feature Engineering")
print("    ✅ Phase 3: Model Building (5 models, default params)")
print("    ✅ Phase 4: Model Evaluation")
print("    ✅ Phase 4.2: Hyperparameter Tuning (LightGBM + XGBoost)")
print("    ✅ Phase 5: Final Report")
print("    ✅ Phase 5.2: Final Report — Tuning (notebook này)")
print(f"\n  Best Model: LightGBM Tuned (AUC-ROC = {best['AUC-ROC']:.4f}, Recall = {best['Recall']:.4f})")
print("\n" + "═" * 80)


════════════════════════════════════════════════════════════════════════════════
   DỰ ÁN HOÀN THÀNH — Give Me Some Credit — Credit Scoring
════════════════════════════════════════════════════════════════════════════════

  Thời gian: 2026-03-13 20:33:30

  Phases đã thực hiện:
    ✅ Phase 1: EDA
    ✅ Phase 2: Preprocessing & Feature Engineering
    ✅ Phase 3: Model Building (5 models, default params)
    ✅ Phase 4: Model Evaluation
    ✅ Phase 4.2: Hyperparameter Tuning (LightGBM + XGBoost)
    ✅ Phase 5: Final Report
    ✅ Phase 5.2: Final Report — Tuning (notebook này)

  Best Model: LightGBM Tuned (AUC-ROC = 0.8695, Recall = 0.7900)

════════════════════════════════════════════════════════════════════════════════
